## **비정형 문서 로더 비교**: Unstructured vs Docling vs LlamaParse

- 현대의 AI 애플리케이션에서 PDF, DOCX, PPTX 등의 **복잡한 문서를 정확하게 파싱**하는 것은 매우 중요
- 특히 RAG(Retrieval-Augmented Generation) 시스템에서는 **문서의 구조와 내용을 정확히 이해하고 추출**하는 능력이 성능을 크게 좌우

<br/>

- **종합 비교표**

   | 항목 | Unstructured | Docling | LlamaParse |
   |------|-------------|---------|------------|
   | **라이선스** | 오픈소스 + 상용 | MIT (완전 무료) | 상용 (무료 플랜) |
   | **파일 형식** | 64+ 형식 | 주요 형식 지원 | 10+ 형식 |
   | **처리 속도** | 느림 (50p: 141초) | 중간 (선형 확장) | 매우 빠름 (6초) |
   | **정확도** | 75-100% | 97.9% | 중간 |
   | **테이블 추출** | 단순: 100%, 복잡: 75% | 97.9% | 개선 필요 |
   | **OCR 지원** | ✅ | ✅ | ✅ |
   | **로컬 실행** | ✅ | ✅ | ❌ (API만) |
   | **다국어 지원** | ✅ | ✅ | ✅ |
   | **LangChain 통합** | ✅ | ✅ | ✅ |
   | **LlamaIndex 통합** | ✅ | ✅ | ✅ 네이티브 |

<br/>

- 🏆 **최고 정확도가 필요한 경우 → Docling**
   - 비즈니스 크리티컬한 금융 보고서 분석
   - 복잡한 테이블이 포함된 학술 논문 처리
   - 법률 문서의 정확한 구조 분석
   - 링크: https://github.com/docling-project/docling

- ⚡ **최고 속도가 필요한 경우 → LlamaParse**
   - 실시간 문서 처리 시스템
   - 대량의 문서를 빠르게 처리하는 배치 작업
   - 프로토타이핑 및 초기 개발 단계
   - 링크: https://www.llamaindex.ai/llamaparse

- 🔧 **최고 유연성이 필요한 경우 → Unstructured**
   - 다양한 문서 형식을 처리하는 통합 시스템
   - 복잡한 ETL 파이프라인 구축
   - 엔터프라이즈급 데이터 처리 플랫폼
   - 링크: https://github.com/Unstructured-IO/unstructured

<br/>

- **[출처]**: [PDF Data Extraction Benchmark 2025: Comparing Docling, Unstructured, and LlamaParse for Document Processing Pipelines](https://procycons.com/en/blogs/pdf-data-extraction-benchmark/)

---

### **필요한 PDF 파일 안내**

이 노트북 실습을 위해 다음 PDF 파일들이 필요합니다:

1. **labor_law.pdf** - 근로기준법 문서
2. **transformer.pdf** - "Attention Is All You Need" Transformer 논문
   - 출처: https://arxiv.org/pdf/1706.03762.pdf
3. **tsla-20241231-gen.pdf** - Tesla 10-K 연차 보고서
   - 출처: https://ir.tesla.com/#quarterly-disclosure

위 파일들을 `data/` 폴더에 다운로드하여 준비해주세요.

---
## **환경 설정 및 준비**
`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob
from pathlib import Path

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

`(3) Langsmith tracing 설정`

In [ ]:
# Langsmith tracing 여부를 확인 (true: langsmith 추적 활성화, false: langsmith 추적 비활성화)
# import os
# print(os.getenv('LANGSMITH_TRACING'))

---

## **Docling을 활용한 PDF 문서 처리 단계별 가이드**

- Docling은 PDF, DOCX, HTML 등 다양한 문서 형식을 구조화된 데이터로 변환하는 라이브러리
-  PDF 문서 처리를 중심으로 단계별로 학습


- **Docling 설치**

   ```bash
   # pip 설치
   pip install docling
   ```

   ```bash
   # uv 설치
   uv add docling
   ```

### **1. 기본 PDF 변환**

In [ ]:
from docling.document_converter import DocumentConverter

# DocumentConverter 초기화 (기본 설정)
converter = DocumentConverter()

# PDF 파일 변환
pdf_path = "data/labor_law.pdf"  # 근로기준법 문서 경로

try:
    # 변환 실행
    result = converter.convert(pdf_path)
    
    # 변환 결과 확인
    if result.status.name in ['SUCCESS', 'PARTIAL_SUCCESS']:
        print(f"변환 성공: {result.status.name}")
        print(f"문서 제목: {result.document.name}")

    else:
        print(f"변환 실패: {result.status}")
        
except Exception as e:
    print(f"오류 발생: {e}")

In [ ]:
type(result)  # ConversionResult 타입 확인

In [ ]:
result.model_dump().keys()  # ConversionResult의 속성 키 확인

### **2. 텍스트 추출 및 마크다운 변환**

In [ ]:
# DoclingDocument 객체로 변환
document = result.document

In [3]:
document.model_dump()  # DoclingDocument의 속성 확인

NameError: name 'document' is not defined

In [ ]:
# 마크다운으로 내보내기
markdown_text = document.export_to_markdown()

print(markdown_text[:500] + "...")  # 처음 500자만 출력

In [ ]:
# 순수 텍스트로 내보내기
plain_text = document.export_to_text()
print(f"{len(plain_text)}자")

print(plain_text[:300] + "...")  # 처음 300자만 출력

In [ ]:
# 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "labor_law.txt", "w", encoding="utf-8") as f:
    f.write(plain_text)

### **3. 고급 설정 (OCR, 테이블 구조)**

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
    TableStructureOptions
)

class AdvancedDocProcessor:
    """고급 문서 처리기"""
    
    def __init__(self, enable_ocr=False, enable_table_structure=True):
        """
        Args:
            enable_ocr: OCR 기능 활성화 (스캔된 문서용)
            enable_table_structure: 테이블 구조 분석 활성화
        """
        
        # 파이프라인 옵션 설정
        pipeline_options = PdfPipelineOptions()  # PDF 변환을 위한 파이프라인 옵션 (기본값 사용)
        pipeline_options.do_ocr = enable_ocr   # OCR 활성화 여부
        pipeline_options.do_table_structure = enable_table_structure # 테이블 구조 분석 활성화 여부 
        
        # 테이블 구조 분석 세부 설정
        if enable_table_structure:
            pipeline_options.table_structure_options.do_cell_matching = True # 셀 매칭
        
        # CPU 사용 설정 (GPU가 없는 환경)
        pipeline_options.accelerator_options = AcceleratorOptions(
            device=AcceleratorDevice.CPU
        )
        
        # DocumentConverter 초기화
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options
                )
            }
        )
        
        print(f"🔧 처리기 초기화 완료:")
        print(f"   - OCR: {'활성화' if enable_ocr else '비활성화'}")
        print(f"   - 테이블 구조 분석: {'활성화' if enable_table_structure else '비활성화'}")
    
    def process_pdf(self, pdf_path):
        """PDF 처리"""
        try:
            print(f"처리 시작: {pdf_path}")
            result = self.converter.convert(str(pdf_path))
            
            if result.status.name in ['SUCCESS', 'PARTIAL_SUCCESS']:
                print(f"처리 완료: {result.status.name}")
                return result.document
            else:
                print(f"처리 실패: {result.status}")
                return None
                
        except Exception as e:
            print(f"오류 발생: {e}")
            return None

`(1) 텍스트만 추출`

In [ ]:
# 기본 설정으로 처리
basic_processor = AdvancedDocProcessor(
    enable_ocr=False,
    enable_table_structure=False
)

# PDF 처리 실행
pdf_path = "data/transformer.pdf"  # 변환할 PDF 문서 경로 
document = basic_processor.process_pdf(pdf_path)

In [ ]:
# 결과 분석
markdown = document.export_to_markdown()

print(f"- 문서명: {document.name}")
print(f"- 텍스트 길이: {len(markdown)}자")

# 테이블이 있는지 확인
if "| " in markdown or "|--" in markdown:
    print("   - 🔍 테이블 구조 감지됨")
else:
    print("   - 📝 일반 텍스트 문서")

In [ ]:
# 마크다운 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "transformer_analysis.md", "w", encoding="utf-8") as f:
    f.write(markdown)

`(2) 테이블 구조 추출`

- **테슬라 10-K 보고서** PDF 파일을 **인터넷에서 다운로드**

- **SEC EDGAR 웹사이트** 또는 **테슬라 IR 페이지**에서 다운로드 가능

- 출처: https://ir.tesla.com/#quarterly-disclosure

In [ ]:
# 고급 설정으로 처리
ocr_processor = AdvancedDocProcessor(
    enable_ocr=True,  # OCR 활성화
    enable_table_structure=True # 테이블 구조 분석 활성화
)

# PDF 처리 실행
pdf_path = "data/tsla-20241231-gen.pdf"  # 변환할 PDF 문서 경로 
document = ocr_processor.process_pdf(pdf_path)

In [ ]:
# 결과 분석
markdown = document.export_to_markdown()

print(f"- 문서명: {document.name}")
print(f"- 텍스트 길이: {len(markdown)}자")

# 테이블이 있는지 확인
if "| " in markdown or "|--" in markdown:
    print("   - 🔍 테이블 구조 감지됨")
else:
    print("   - 📝 일반 텍스트 문서")

In [ ]:
# 마크다운 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "tsla_analysis_ocr.md", "w", encoding="utf-8") as f:
    f.write(markdown)

### **4. 문서 구조 분석**

In [ ]:
# DoclingDocument의 속성 확인
document.model_dump().keys()  

In [ ]:
# DoclingDocument의 텍스트 내용 확인
document.texts

In [ ]:
# 첫 번째 텍스트 아이템의 속성 확인
document.texts[0].model_dump()  

In [ ]:
# DoclingDocument의 테이블 구조 확인
document.tables

In [ ]:
# 첫 번째 테이블 아이템의 속성 확인
document.tables[0].model_dump() 

In [ ]:
# DoclingDocument의 텍스트와 테이블 요소를 순서대로 JSON으로 변환
from docling_core.types.doc import TextItem, TableItem
import pandas as pd

def convert_document_to_ordered_json(document):
    """문서 요소를 원본 순서대로 JSON 배열로 변환 (테이블은 마크다운+딕셔너리 형식)"""
    elements = []
    
    for item, level in document.iterate_items():
        if isinstance(item, TextItem):
            elements.append({
                "type": "text",
                "content": item.text.replace("\t", " ") or "",
                "page": item.prov[0].page_no if item.prov else None,
                "label": item.label.value if item.label else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
        elif isinstance(item, TableItem):
            table_content = {}
            
            try:
                # DataFrame으로 변환 (document 객체 전달 필요)
                df = item.export_to_dataframe(doc=document)

                # 마크다운 형식으로 변환
                markdown_content = df.to_markdown(index=False)  # 인덱스 제외

                # 딕셔너리 형식으로 변환 (여러 옵션 제공)
                dict_content = df.to_dict('records')     # 각 행을 딕셔너리로

                table_content = {
                    "markdown": markdown_content,
                    "data": dict_content,
                    "status": "success"
                }
                
            except Exception as e:
                # DataFrame 변환이 실패한 경우 대안 시도
                try:
                    html_content = item.export_to_html()
                    table_content = {
                        "html": html_content,
                        "status": "html_fallback",
                        "error": str(e)
                    }
                except Exception as e2:
                    table_content = {
                        "status": "failed",
                        "error": f"DataFrame 변환 실패: {str(e)}, HTML 변환 실패: {str(e2)}"
                    }
            
            elements.append({
                "type": "table",
                "content": table_content,
                "page": item.prov[0].page_no if item.prov else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
    
    return elements

# 문서 요소를 원본 순서대로 JSON 배열로 변환
ordered_json = convert_document_to_ordered_json(document)
print(f"문서 요소를 원본 순서대로 JSON 배열로 변환 완료. 총 {len(ordered_json)}개 요소.")

In [ ]:
ordered_json[:10]

In [ ]:
# 표 객체 확인
table_indices = []
for i, item in enumerate(ordered_json):
    if item['type'] == 'table':
        table_indices.append(i)

print(len(table_indices))
print(table_indices)


In [ ]:
# 397번째 요소의 내용 확인 (표)
print(ordered_json[397]['content']['markdown']  )

In [ ]:
# 397번째 요소의 내용 확인 (표)
pd.DataFrame(ordered_json[397]['content']['data'])

In [ ]:
# pickle로 저장
import pickle

save_folder = Path("data/docling_output")
pickle_path = save_folder / "tsla_analysis_ocr.pkl"

with open(pickle_path, "wb") as f:
    pickle.dump(ordered_json, f)

print(f"pickle 파일로 저장됨: {pickle_path}")

# [실습] 

- transformer.pdf 파일 처리
- 테이블 요소와 텍스트 요소를 각각 구분하여 정리
    - 테이블 요소는 마크다운 형식으로 저장
    - 텍스트 요소는 텍스트 형식으로 저장

In [ ]:
# 여기에 코드를 작성하세요.

<details close>
<summary>💡 정답 확인</summary>

```python
# 1. transformer.pdf 파일 처리
processor = AdvancedDocProcessor(
    enable_ocr=False,
    enable_table_structure=True
)

pdf_path = "data/transformer.pdf"
document = processor.process_pdf(pdf_path)

# 2. 문서 요소를 원본 순서대로 JSON 배열로 변환
ordered_json = convert_document_to_ordered_json(document)
print(f"총 {len(ordered_json)}개 요소 추출 완료")

# 3. 테이블 요소와 텍스트 요소 분리
table_elements = []
text_elements = []

for item in ordered_json:
    if item['type'] == 'table':
        table_elements.append(item)
    elif item['type'] == 'text':
        text_elements.append(item)

print(f"테이블 요소: {len(table_elements)}개")
print(f"텍스트 요소: {len(text_elements)}개")

# 4. 테이블 요소를 마크다운 형식으로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "transformer_tables.md", "w", encoding="utf-8") as f:
    f.write("# Transformer 논문 - 테이블 추출 결과\n\n")
    
    for i, table in enumerate(table_elements, 1):
        f.write(f"## 테이블 {i}\n")
        f.write(f"- 페이지: {table['page']}\n")
        f.write(f"- 레벨: {table['level']}\n\n")
        
        if table['content']['status'] == 'success':
            f.write(table['content']['markdown'])
        else:
            f.write(f"(테이블 추출 실패: {table['content'].get('error', 'Unknown error')})")
        
        f.write("\n\n---\n\n")

print(f"테이블 마크다운 파일 저장 완료: {save_folder / 'transformer_tables.md'}")

# 5. 텍스트 요소를 텍스트 형식으로 저장
with open(save_folder / "transformer_text.txt", "w", encoding="utf-8") as f:
    f.write("# Transformer 논문 - 텍스트 추출 결과\n\n")
    
    for item in text_elements:
        if item['content'].strip():  # 빈 텍스트 제외
            # 라벨과 페이지 정보 추가
            if item['label']:
                f.write(f"[{item['label']}] ")
            f.write(item['content'])
            f.write("\n\n")

print(f"텍스트 파일 저장 완료: {save_folder / 'transformer_text.txt'}")

# 6. 결과 요약 출력
print("\n=== 처리 결과 요약 ===")
print(f"문서명: {document.name}")
print(f"총 요소 수: {len(ordered_json)}개")
print(f"  - 테이블: {len(table_elements)}개")
print(f"  - 텍스트: {len(text_elements)}개")
print(f"\n저장 파일:")
print(f"  - {save_folder / 'transformer_tables.md'}")
print(f"  - {save_folder / 'transformer_text.txt'}")
```

</details>
